# 훈련 추천 지표 준비

일별 훈련 데이터를 이용하여 오늘의 훈련 추천에 사용할 후보 지표를 만든다.

## 1. 추천 시점과 입력 정보

오늘 아침의 훈련을 추천한다고 가정한다. 추천 시점에는 오늘의 운동 결과를 알 수 없으므로, 추천일 이전까지 기록된 데이터만 사용한다.

추천 지표 후보는 다음과 같다.

- 직전 7일 누적 TSS
- 직전 28일 누적 TSS
- 단기 부하와 장기 부하의 관계
- 전날까지의 연속 라이드 일수
- 최근 무기록 일수
- 직전 7일 운동 시간과 라이드 횟수

TSS는 기록된 훈련 부하를 요약한 값이며, 실제 피로나 회복 상태를 직접 측정하는 값은 아니다. 또한 라이드 무기록일을 실제 휴식일로 단정하지 않는다.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()
processed_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_bike_rides_cleaned.csv"
)

rides_df = pd.read_csv(
    processed_path,
    parse_dates=["date"],
)

analysis_period_df = rides_df.loc[
    (rides_df["date"] >= "2009-02-01")
    & (rides_df["date"] < "2009-07-01")
].copy()

analysis_period_df.shape

(111, 63)

In [5]:
daily_training_df = (
    analysis_period_df
    .set_index("date")
    .resample("D")
    .agg(
        workout_hours=("workout_hours", "sum"),
        tss=("coggan_tss", "sum"),
        ride_count=("sport", "size"),
    )
)

daily_training_df.head(10)

,workout_hours,tss,ride_count
date,,,
2009-02-07 00:00:00+00:00,1.193333,111.22867,1
2009-02-08 00:00:00+00:00,2.460833,176.54760,1
2009-02-09 00:00:00+00:00,0.000000,0.00000,0
2009-02-10 00:00:00+00:00,0.833333,43.90424,1
2009-02-11 00:00:00+00:00,1.000000,50.11669,1
2009-02-12 00:00:00+00:00,2.000000,100.23338,2
2009-02-13 00:00:00+00:00,1.124167,108.21922,1
2009-02-14 00:00:00+00:00,1.970278,116.20702,3
2009-02-15 00:00:00+00:00,1.962222,186.06192,1


In [6]:
daily_training_df["has_recorded_ride"] = (
    daily_training_df["ride_count"] > 0
)

daily_training_df["has_recorded_ride"].value_counts()

has_recorded_ride
True     93
False    49
Name: count, dtype: int64

In [10]:
daily_training_df["previous_7_day_tss"] = (
    daily_training_df["tss"]
    .shift(1)
    .rolling(window=7, min_periods=7)
    .sum()
)

daily_training_df[
    ["tss", "previous_7_day_tss"]
].head(10).round(2)

,tss,previous_7_day_tss
date,,
2009-02-07 00:00:00+00:00,111.23,NaN
2009-02-08 00:00:00+00:00,176.55,NaN
2009-02-09 00:00:00+00:00,0.00,NaN
2009-02-10 00:00:00+00:00,43.90,NaN
2009-02-11 00:00:00+00:00,50.12,NaN
2009-02-12 00:00:00+00:00,100.23,NaN
2009-02-13 00:00:00+00:00,108.22,NaN
2009-02-14 00:00:00+00:00,116.21,590.25
2009-02-15 00:00:00+00:00,186.06,595.23


In [11]:
daily_training_df["previous_28_day_tss"] = (
    daily_training_df["tss"]
    .shift(1)
    .rolling(window=28, min_periods=28)
    .sum()
)

daily_training_df[
    ["previous_7_day_tss", "previous_28_day_tss"]
].loc["2009-03-05":"2009-03-09"].round(2)

,previous_7_day_tss,previous_28_day_tss
date,,
2009-03-05 00:00:00+00:00,776.14,NaN
2009-03-06 00:00:00+00:00,676.18,NaN
2009-03-07 00:00:00+00:00,531.81,2494.51
2009-03-08 00:00:00+00:00,582.35,2511.41
2009-03-09 00:00:00+00:00,348.77,2334.86


In [12]:
daily_training_df["previous_28_day_tss_weekly_average"] = (
    daily_training_df["previous_28_day_tss"] / 4
)

daily_training_df["short_to_long_tss_ratio"] = (
    daily_training_df["previous_7_day_tss"]
    / daily_training_df["previous_28_day_tss_weekly_average"]
)

daily_training_df[
    [
        "previous_7_day_tss",
        "previous_28_day_tss_weekly_average",
        "short_to_long_tss_ratio",
    ]
].loc["2009-03-07":"2009-03-11"].round(2)

,previous_7_day_tss,previous_28_day_tss_weekly_average,short_to_long_tss_ratio
date,,,
2009-03-07 00:00:00+00:00,531.81,623.63,0.85
2009-03-08 00:00:00+00:00,582.35,627.85,0.93
2009-03-09 00:00:00+00:00,348.77,583.72,0.60
2009-03-10 00:00:00+00:00,348.92,612.11,0.57
2009-03-11 00:00:00+00:00,348.92,601.14,0.58


In [13]:
daily_training_df["previous_7_day_workout_hours"] = (
    daily_training_df["workout_hours"]
    .shift(1)
    .rolling(window=7, min_periods=7)
    .sum()
)

daily_training_df[
    ["workout_hours", "previous_7_day_workout_hours"]
].head(10).round(2)

,workout_hours,previous_7_day_workout_hours
date,,
2009-02-07 00:00:00+00:00,1.19,NaN
2009-02-08 00:00:00+00:00,2.46,NaN
2009-02-09 00:00:00+00:00,0.00,NaN
2009-02-10 00:00:00+00:00,0.83,NaN
2009-02-11 00:00:00+00:00,1.00,NaN
2009-02-12 00:00:00+00:00,2.00,NaN
2009-02-13 00:00:00+00:00,1.12,NaN
2009-02-14 00:00:00+00:00,1.97,8.61
2009-02-15 00:00:00+00:00,1.96,9.39


In [14]:
daily_training_df["previous_7_day_ride_count"] = (
    daily_training_df["ride_count"]
    .shift(1)
    .rolling(window=7, min_periods=7)
    .sum()
)

daily_training_df[
    ["ride_count", "previous_7_day_ride_count"]
].head(10)

,ride_count,previous_7_day_ride_count
date,,
2009-02-07 00:00:00+00:00,1,NaN
2009-02-08 00:00:00+00:00,1,NaN
2009-02-09 00:00:00+00:00,0,NaN
2009-02-10 00:00:00+00:00,1,NaN
2009-02-11 00:00:00+00:00,1,NaN
2009-02-12 00:00:00+00:00,2,NaN
2009-02-13 00:00:00+00:00,1,NaN
2009-02-14 00:00:00+00:00,3,7.0
2009-02-15 00:00:00+00:00,1,9.0


In [15]:
daily_training_df["ride_status_changed"] = (
    daily_training_df["has_recorded_ride"]
    .ne(daily_training_df["has_recorded_ride"].shift())
)

daily_training_df["ride_status_streak_id"] = (
    daily_training_df["ride_status_changed"]
    .cumsum()
)

daily_training_df[
    [
        "has_recorded_ride",
        "ride_status_changed",
        "ride_status_streak_id",
    ]
].head(10)

,has_recorded_ride,ride_status_changed,ride_status_streak_id
date,,,
2009-02-07 00:00:00+00:00,True,True,1
2009-02-08 00:00:00+00:00,True,False,1
2009-02-09 00:00:00+00:00,False,True,2
2009-02-10 00:00:00+00:00,True,True,3
2009-02-11 00:00:00+00:00,True,False,3
2009-02-12 00:00:00+00:00,True,False,3
2009-02-13 00:00:00+00:00,True,False,3
2009-02-14 00:00:00+00:00,True,False,3
2009-02-15 00:00:00+00:00,True,False,3


In [16]:
daily_training_df["recorded_ride_streak_ending_today"] = (
    daily_training_df["has_recorded_ride"]
    .groupby(daily_training_df["ride_status_streak_id"])
    .cumsum()
)

daily_training_df[
    [
        "has_recorded_ride",
        "ride_status_streak_id",
        "recorded_ride_streak_ending_today",
    ]
].head(10)

,has_recorded_ride,ride_status_streak_id,recorded_ride_streak_ending_today
date,,,
2009-02-07 00:00:00+00:00,True,1,1
2009-02-08 00:00:00+00:00,True,1,2
2009-02-09 00:00:00+00:00,False,2,0
2009-02-10 00:00:00+00:00,True,3,1
2009-02-11 00:00:00+00:00,True,3,2
2009-02-12 00:00:00+00:00,True,3,3
2009-02-13 00:00:00+00:00,True,3,4
2009-02-14 00:00:00+00:00,True,3,5
2009-02-15 00:00:00+00:00,True,3,6


In [17]:
daily_training_df["previous_day_recorded_ride_streak_days"] = (
    daily_training_df["recorded_ride_streak_ending_today"]
    .shift(1)
)

daily_training_df[
    [
        "recorded_ride_streak_ending_today",
        "previous_day_recorded_ride_streak_days",
    ]
].head(10)

,recorded_ride_streak_ending_today,previous_day_recorded_ride_streak_days
date,,
2009-02-07 00:00:00+00:00,1,NaN
2009-02-08 00:00:00+00:00,2,1.0
2009-02-09 00:00:00+00:00,0,2.0
2009-02-10 00:00:00+00:00,1,0.0
2009-02-11 00:00:00+00:00,2,1.0
2009-02-12 00:00:00+00:00,3,2.0
2009-02-13 00:00:00+00:00,4,3.0
2009-02-14 00:00:00+00:00,5,4.0
2009-02-15 00:00:00+00:00,6,5.0


In [18]:
daily_training_df["no_record_streak_ending_today"] = (
    (~daily_training_df["has_recorded_ride"])
    .groupby(daily_training_df["ride_status_streak_id"])
    .cumsum()
)

daily_training_df[
    [
        "has_recorded_ride",
        "no_record_streak_ending_today",
    ]
].head(10)

,has_recorded_ride,no_record_streak_ending_today
date,,
2009-02-07 00:00:00+00:00,True,0
2009-02-08 00:00:00+00:00,True,0
2009-02-09 00:00:00+00:00,False,1
2009-02-10 00:00:00+00:00,True,0
2009-02-11 00:00:00+00:00,True,0
2009-02-12 00:00:00+00:00,True,0
2009-02-13 00:00:00+00:00,True,0
2009-02-14 00:00:00+00:00,True,0
2009-02-15 00:00:00+00:00,True,0


In [19]:
daily_training_df["previous_day_no_record_streak_days"] = (
    daily_training_df["no_record_streak_ending_today"]
    .shift(1)
)

daily_training_df[
    [
        "no_record_streak_ending_today",
        "previous_day_no_record_streak_days",
    ]
].head(10)

,no_record_streak_ending_today,previous_day_no_record_streak_days
date,,
2009-02-07 00:00:00+00:00,0,NaN
2009-02-08 00:00:00+00:00,0,0.0
2009-02-09 00:00:00+00:00,1,0.0
2009-02-10 00:00:00+00:00,0,1.0
2009-02-11 00:00:00+00:00,0,0.0
2009-02-12 00:00:00+00:00,0,0.0
2009-02-13 00:00:00+00:00,0,0.0
2009-02-14 00:00:00+00:00,0,0.0
2009-02-15 00:00:00+00:00,0,0.0


In [20]:
recommendation_input_columns = [
    "previous_7_day_tss",
    "previous_28_day_tss",
    "short_to_long_tss_ratio",
    "previous_7_day_workout_hours",
    "previous_7_day_ride_count",
    "previous_day_recorded_ride_streak_days",
    "previous_day_no_record_streak_days",
]

recommendation_inputs_df = daily_training_df[
    recommendation_input_columns
].copy()

recommendation_inputs_df.loc[
    "2009-03-07":"2009-03-11"
].round(2)

,previous_7_day_tss,previous_28_day_tss,short_to_long_tss_ratio,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days
date,,,,,,,
2009-03-07 00:00:00+00:00,531.81,2494.51,0.85,7.54,4.0,1.0,0.0
2009-03-08 00:00:00+00:00,582.35,2511.41,0.93,7.57,4.0,2.0,0.0
2009-03-09 00:00:00+00:00,348.77,2334.86,0.60,4.37,3.0,0.0,1.0
2009-03-10 00:00:00+00:00,348.92,2448.45,0.57,3.81,3.0,1.0,0.0
2009-03-11 00:00:00+00:00,348.92,2404.54,0.58,3.81,3.0,0.0,1.0


In [21]:
manual_previous_7_day_tss = (
    daily_training_df
    .loc["2009-03-03":"2009-03-09", "tss"]
    .sum()
)

stored_previous_7_day_tss = (
    recommendation_inputs_df
    .loc["2009-03-10", "previous_7_day_tss"]
)

pd.Series(
    {
        "manual_previous_7_day_tss": manual_previous_7_day_tss,
        "stored_previous_7_day_tss": stored_previous_7_day_tss,
        "difference": (
            manual_previous_7_day_tss
            - stored_previous_7_day_tss
        ),
    }
).round(2)

manual_previous_7_day_tss    348.92
stored_previous_7_day_tss    348.92
difference                     0.00
dtype: float64

In [22]:
manual_previous_28_day_tss = (
    daily_training_df
    .loc["2009-02-10":"2009-03-09", "tss"]
    .sum()
)

stored_previous_28_day_tss = (
    recommendation_inputs_df
    .loc["2009-03-10", "previous_28_day_tss"]
)

pd.Series(
    {
        "manual_previous_28_day_tss": manual_previous_28_day_tss,
        "stored_previous_28_day_tss": stored_previous_28_day_tss,
        "difference": (
            manual_previous_28_day_tss
            - stored_previous_28_day_tss
        ),
    }
).round(2)

manual_previous_28_day_tss    2448.45
stored_previous_28_day_tss    2448.45
difference                       0.00
dtype: float64

In [25]:
recommendation_ready_df = (
    recommendation_inputs_df
    .dropna()
    .copy()
)

pd.Series(
    {
        "row_count": len(recommendation_ready_df),
        "first_date": recommendation_ready_df.index.min(),
        "last_date": recommendation_ready_df.index.max(),
    }
)

row_count                           114
first_date    2009-03-07 00:00:00+00:00
last_date     2009-06-28 00:00:00+00:00
dtype: object